# SRM + GBRAS-Net (arm G)

Our fixed SRM90 and unchanged [-3,3] clamp -> learned Conv1x1 90->30 ->
the authors' GBRAS-Net body at commit 0b40ab046116beb64dc1b0973546cf9a197b883e.
Paper: Tabares-Soto (Reinel) et al., IEEE Access 2021,
doi:10.1109/ACCESS.2021.3052494. See cvdl/gbras.py for the layer/count audit
and the unavailable-paper-PDF limit. No GBRAS 3*tanh or extra body pool.

The body includes its published beta-only BatchNorm (Torch momentum .8,
eps .001): norm=batch records this intrinsic choice; it is not optional.
Training uses the SRM-30 recipe below, not the authors' Adam/100-epoch recipe.
Batch128 and 30 epochs remain provisional until the PC pilot and D3/D4 decision.
Use the approved pilot=true path separately; do not launch training or G4 yet.
D5: best train-matched-validation checkpoint, recipe60da4391043e8ce48ff2.
D6/decision: >=.03 pooled hard-k1-low AUC gain with paired95% CI above0,
plus the standard-256 guard; compare to baseline eval11a82bf812e573c7c62d.

Training is UNPAIRED (128 images): the OTF stream shuffles clean/stego items.
The authors also use shuffled individual images (Keras model.fit default),
not guaranteed pairs. Turning paired_batches on would normalize combined
clean+stego batches of2B correlated images, changing the sampled BN statistics,
not its equations. Validation concatenates pairs in eval mode: it must NOT
update BN or use batch statistics. Deployment-style inference also needs eval.

Native class-score maps have16px stride. Diagnostic-only2x2 averaging gives
32px cells, preserves mean logits, and is labelled class_score, NOT a CAM.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "cvdl").is_dir())
sys.path.insert(0, str(ROOT))
from cvdl.settings import load

settings = load(
    name="srm-gbras",
    front_end="srm",
    tile_size=512,
    train_tile_size=256,
    train_grid="overlap_last",
    batch_size=128,
    paired_batches=False,
    tiles_per_epoch=180000,
    post_fit_eval="none",
    model='gbras',
    norm='batch',
    clip_t=3.0,
    lr=0.0003,
    warmup_epochs=1,
    grad_clip=1.0,
    lr_drops=((5, 0.3), (20, 0.1)),
    epochs=30,
    train_yaml='configs/train-k1to4.yaml',
    data='otf',
    source='decode',
    source_mix={'raise': 0.4, 'lsdir': 0.2, 'alaska': 0.4},
    evaluation_checkpoint='best',
    checkpoint_validation='train_matched',
    validation_recipe='60da4391043e8ce48ff2',
    val_cap_raise=5000,
    val_cap_lsdir=1500,
    extra_tests='test-bpp1,source-panels',
)
# Compilation is machine-owned: G4 defaults to default; PC pilot uses --set compile=default.
settings.apply_runtime_environment()

## Data and machine setup

Keep the standard test unchanged. QUICK limits normal evaluation and extra
panels. Data identities and settings are saved beside the checkpoints.

In [ ]:
import torch
from cvdl import runtime, data, evaluation, diagnostics
from cvdl.lit import SteganalysisModule
from cvdl.metrics import metrics
from lsb_torch import FixedPairs

runtime.configure(settings)
RUN_NAME, QUICK, V3_DATA = settings.run_name, settings.quick, settings.data
EPOCHS, BATCH_SIZE, COMPILE_MODE = settings.epochs, settings.batch_size, settings.compile
DEVICE = torch.device(settings.device)
otf_data, tables = runtime.create_data(settings)
print(f"Run: {RUN_NAME}; device={DEVICE}; epochs={EPOCHS}; batch={BATCH_SIZE}; QUICK={QUICK}")

In [ ]:
loaders, clean_batch, stego_batch = runtime.create_loaders(settings, otf_data, tables)
train_loader, val_loader = loaders.train, loaders.val
test_loader, train_eval_loader = loaders.test, loaders.train_eval
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

## Model checks and training

The standard checks verify that SRM remains frozen. Best-on-validation and
final-last checkpoints follow the existing training path. Compilation is
in-place on net, so checkpoint keys stay eager/reloadable.


In [ ]:
runtime.model_checks(settings, clean_batch, stego_batch)

In [ ]:
bundle = runtime.create_training(settings, otf_data, loaders)
trainer, lightning_model = bundle.trainer, bundle.lightning_model
checkpoint_dir, checkpoint_callback = bundle.checkpoint_dir, bundle.checkpoint_callback
assert trainer.checkpoint_callbacks == [checkpoint_callback]

In [ ]:
trainer.fit(lightning_model, train_loader, val_loader, ckpt_path="last")
best_checkpoint = runtime.finish_training(settings, bundle, loaders)

## Predictions and results

Reload the best eager model for float32 evaluation. Save paired predictions,
print P_E/AUC summaries, run selected extra panels, and write the per-setting
CSV plus four breakdown heatmaps. Batch checks use copies of the best network.

In [ ]:
if settings.post_fit_eval == "full":
    best_model = SteganalysisModule.load_from_checkpoint(best_checkpoint, map_location=DEVICE, weights_only=False)
    best_model.to(DEVICE)
    train_predictions = evaluation.predict_pairs(best_model, train_eval_loader, device=DEVICE, data_mode=V3_DATA)
    val_predictions = evaluation.predict_pairs(best_model, val_loader, device=DEVICE, data_mode=V3_DATA)
    predictions = evaluation.predict_pairs(best_model, test_loader, device=DEVICE, data_mode=V3_DATA)
    for name, frame in (("train", train_predictions), ("val", val_predictions), ("test", predictions)):
        print(name, metrics(frame))
    print(evaluation.per_setting_table(predictions, data_mode=V3_DATA).to_string(index=False))
    predictions_path = evaluation.save_predictions(predictions, checkpoint_dir / "test_predictions.csv")

In [ ]:
if settings.post_fit_eval == "full":
    extra_results = evaluation.extra_panels(best_checkpoint, root=ROOT, output=checkpoint_dir,
        panels=settings.extra_tests, workers=settings.eval_workers, device=DEVICE, quick=QUICK,
        data_mode=V3_DATA, manifest_root=settings.manifest_root,
        source_panels=getattr(otf_data, "extra_pairs", ()))
    for name, (info, table) in extra_results.items():
        print(f"EXTRA {name}: {info['pairs']} pairs")
        print(table.to_string(index=False))

In [ ]:
if settings.post_fit_eval == "full":
    summary, per_setting, cell_table, heatmap_paths = evaluation.breakdown(predictions_path,
        labels_path=settings.labels_path if V3_DATA == "png" else None, show=True)
    print("TEST SUMMARY:", summary)
    print(per_setting.to_string(index=False))
    assert len(heatmap_paths) == 4
    print("OUTPUT CHECK: predictions, per-setting CSV and four heatmaps written")

In [ ]:
if settings.post_fit_eval == "full":
    diagnostic_results = {}
    for split_name in ("train", "val"):
        if V3_DATA == "otf":
            base = otf_data.fixed_train if split_name == "train" else FixedPairs(otf_data.corpus, "val", settings.eval_profile)
        else:
            base = data.PairDataset(tables.pair_splits.loc[tables.pair_splits["split"] == split_name], settings.data_root)
        selected = diagnostics.diagnostic_pairs(base, quick=QUICK, data_mode=V3_DATA,
            workers=settings.diag_workers, data_root=settings.data_root)
        if otf_data is not None:
            otf_data.record.setdefault("diagnostic_pair_ids", {})[split_name] = selected.frame["cover_id"].tolist()
        diagnostic_results[split_name] = diagnostics.batch_dependence(best_model.net, selected,
            device=DEVICE, norm=settings.norm, split_name=split_name, quick=QUICK,
            loader_workers=6 if V3_DATA == "png" else 0)
        print(f"CHECK {split_name}:", diagnostic_results[split_name])
    if otf_data is not None:
        otf_data.save_record(checkpoint_dir / "data_run_settings.json")
    settings.write(checkpoint_dir)
    print(f"Saved run: {checkpoint_dir}")